# Training YOLOv8 dari dataset Roboflow (Perception 2, magang PGR)

Jalankan sel dari atas ke bawah. Sebelum mulai: **Runtime > Change runtime type > T4 GPU**.

Alur: unduh dataset dari Roboflow (format YOLOv8) > training > lihat metrik > unduh `best.pt`.

## 1. Cek GPU

In [ ]:
!nvidia-smi

## 2. Instal paket

In [ ]:
!pip install -q ultralytics roboflow

## 3. Unduh dataset dari Roboflow

Di Roboflow: buka **Versions**, pilih versi dataset, **Download Dataset**, format **YOLOv8**, pilih **Show download code**.
Salin nama workspace, nama project, dan nomor versi dari kode itu ke sel di bawah.
API key diketik saat diminta (jangan ditulis di notebook yang dibagikan).

Cara lain tanpa API key: unggah zip hasil ekspor ke panel Files Colab, lalu pakai sel 3b.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

WORKSPACE = "nama-workspace"   # ganti
PROJECT = "nama-project"       # ganti
VERSION = 1                    # ganti

rf = Roboflow(api_key=getpass("Roboflow API key: "))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
DATA_DIR = dataset.location
print(DATA_DIR)

### 3b. (Opsional) Pakai zip yang diunggah manual

In [ ]:
# ZIP = "/content/dataset.zip"   # ganti dengan nama zip yang diunggah
# !unzip -q -o $ZIP -d /content/dataset
# DATA_DIR = "/content/dataset"

## 4. Rapikan `data.yaml`

Ekspor Roboflow menulis path relatif seperti `../train/images`. Sel ini menulis ulang menjadi path absolut supaya training pasti menemukan gambarnya.

In [ ]:
import os, yaml

yaml_path = os.path.join(DATA_DIR, "data.yaml")
cfg = yaml.safe_load(open(yaml_path))
cfg["path"] = DATA_DIR
for split, folder in (("train", "train"), ("val", "valid"), ("test", "test")):
    images = os.path.join(DATA_DIR, folder, "images")
    if os.path.isdir(images):
        cfg[split] = images
    else:
        cfg.pop(split, None)
yaml.safe_dump(cfg, open(yaml_path, "w"), sort_keys=False)
print(open(yaml_path).read())
for folder in ("train", "valid", "test"):
    d = os.path.join(DATA_DIR, folder, "images")
    if os.path.isdir(d):
        print(folder, len(os.listdir(d)), "gambar")

## 5. Training (transfer learning dari `yolov8n.pt`)

50 epoch untuk sekitar 300 gambar butuh beberapa menit di T4. `patience` menghentikan training kalau mAP tidak membaik.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(data=yaml_path, epochs=50, imgsz=640, batch=16, patience=20,
            project="/content/runs", name="roboflow", exist_ok=True)

## 6. Lihat grafik training dan confusion matrix

In [ ]:
from IPython.display import Image, display
RUN = "/content/runs/roboflow"
for f in ("results.png", "confusion_matrix.png"):
    p = os.path.join(RUN, f)
    if os.path.exists(p):
        display(Image(filename=p, width=900))

## 7. Metrik pada data valid (catat untuk laporan)

In [ ]:
from ultralytics import YOLO
best = YOLO(os.path.join(RUN, "weights", "best.pt"))
m = best.val(data=yaml_path, imgsz=640)
print(f"mAP50     = {m.box.map50:.3f}")
print(f"mAP50-95  = {m.box.map:.3f}")
print(f"precision = {m.box.mp:.3f}")
print(f"recall    = {m.box.mr:.3f}")

## 8. Coba prediksi beberapa gambar valid

In [ ]:
import glob
imgs = sorted(glob.glob(os.path.join(DATA_DIR, "valid", "images", "*")))[:4]
for r in best.predict(imgs, conf=0.25, save=True, project="/content/runs", name="prediksi", exist_ok=True):
    display(Image(filename=os.path.join(r.save_dir, os.path.basename(r.path)), width=640))

## 9. Unduh `best.pt`

Simpan `best.pt` ke repo tugas, lalu jalankan di laptop: `python3 deteksi.py --model best.pt --source 0`.

In [ ]:
from google.colab import files
files.download(os.path.join(RUN, "weights", "best.pt"))

## 10. (Opsional) Bandingkan dengan `best_pemateri.pt`

Unggah `best_pemateri.pt` ke panel Files. Perbandingan mAP hanya adil kalau **nama kelasnya sama** dengan dataset kamu.
Kalau kelasnya beda, bandingkan dengan melihat hasil deteksi pada gambar yang sama.

In [ ]:
ref = YOLO("/content/best_pemateri.pt")
print("kelas pemateri:", ref.names)
print("kelas kamu    :", best.names)
if list(ref.names.values()) == list(best.names.values()):
    r = ref.val(data=yaml_path, imgsz=640)
    print(f"pemateri mAP50 = {r.box.map50:.3f}, punyamu = {m.box.map50:.3f}")
else:
    print("kelas berbeda: bandingkan secara visual")
    ref.predict(imgs, conf=0.25, save=True, project="/content/runs", name="prediksi_pemateri", exist_ok=True)